# 🔍 Vérification ANSTAT vs Indicateurs Salaire Brut 2024-2025

**Objectif** : Comparer les min/max par **Grade** et par **Barème / Grille salariale** entre :
- Le fichier de référence `anstat_2025.txt.xlsx` (grille salariale théorique)
- Le fichier `indicateurs_salaire_brut_2024_2025_avec_zeros_SOMMAIRE_200426.xlsx` (indicateurs calculés sur données réelles)

**Étapes** :
1. Charger `anstat_2025.txt.xlsx` et créer la colonne `Montant brut sans indemnités`
2. Calculer les min/max par Grade et par Barème depuis ANSTAT
3. Charger les indicateurs (feuilles GRADE_* et BAREME_G_*)
4. Comparer les bornes et identifier les écarts

## 📦 1. Imports et configuration

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Chemins des fichiers
PATH_ANSTAT = "anstat_2025.txt.xlsx"
PATH_INDICATEURS = "indicateurs_salaire_brut_2024_2025_avec_zeros_SOMMAIRE_200426.xlsx"

print("✅ Imports OK")

✅ Imports OK


## 📂 2. Chargement de `anstat_2025.txt.xlsx`

In [2]:
df_anstat = pd.read_excel(PATH_ANSTAT)
print(f"✅ ANSTAT chargé : {df_anstat.shape[0]} lignes × {df_anstat.shape[1]} colonnes")
print(f"\nColonnes : {df_anstat.columns.tolist()}")
print(f"\nSECTEURS uniques : {df_anstat['SECTEUR'].unique().tolist()}")
print(f"\nGRADES uniques   : {df_anstat['GRADE'].unique().tolist()}")
df_anstat.head(3)

✅ ANSTAT chargé : 754 lignes × 18 colonnes

Colonnes : ['SECTEUR', 'CATEGORIE', 'GRADE', 'INDICE', 'ECHELON/CLASSE', 'Salaire de Base', 'Résidence', 'Logement', 'Transport', 'Allocations\n (2 enfants)', 'AUTRES INDEMNITES', 'Montant brut', 'Charge patronale', 'ITS', 'PENSION', 'MUGEF-CI/CMU', 'NET', 'BASE IMPOSABLE']

SECTEURS uniques : ['Barême Général', 'Informaticien', 'Tech-Santé', 'Cadre-Sup-Santé', 'ENS SEC & PRIM', 'ENS SUP', 'SECRETAIRE DIRECTION', 'GREFFE', 'MAGISTRAT ', 'CORPS PREF', 'CORPS DIPL', 'para militaire']

GRADES uniques   : ['A7', 'A6', 'A5', 'A4', 'A3', 'B3', 'B1', 'C3', 'C2', 'C1', 'D2', 'D1']


,SECTEUR,CATEGORIE,GRADE,INDICE,ECHELON/CLASSE,Salaire de Base,Résidence,Logement,Transport,Allocations\n (2 enfants),AUTRES INDEMNITES,Montant brut,Charge patronale,ITS,PENSION,MUGEF-CI/CMU,NET,BASE IMPOSABLE
0,Barême Général,A,A7,2350,21,548621.60,82293.240,90000,15000,15000,NaN,750914.840,91455.220720,91992.11640,45700.179280,7004.0,606218.544320,630914.840
1,Barême Général,A,A7,2295,33,535781.52,80367.228,90000,15000,15000,NaN,736148.748,89314.779384,88891.23708,44630.600616,7004.0,595622.910304,616148.748
2,Barême Général,A,A7,2255,32,526443.28,78966.492,90000,15000,15000,NaN,725409.772,87758.094776,86636.05212,43852.725224,7004.0,587916.994656,605409.772


## 🧮 3. Création de `Montant brut sans indemnités`

Formule : **Salaire de Base + Résidence + Logement + Transport + Allocations (2 enfants)**

*(On exclut les "AUTRES INDEMNITES" car elles varient selon le poste)*

In [3]:
# Colonnes composant le montant brut sans indemnités
cols_montant_brut_sans_indem = [
    "Salaire de Base",
    "Résidence",
    "Logement",
    "Transport",
    "Allocations\n (2 enfants)"
]

# Vérifier que toutes les colonnes existent
for c in cols_montant_brut_sans_indem:
    assert c in df_anstat.columns, f"Colonne manquante : '{c}'"

# Calculer le montant brut sans indemnités
df_anstat["Montant brut sans indemnités"] = df_anstat[cols_montant_brut_sans_indem].sum(axis=1)

# Réorganiser : insérer après AUTRES INDEMNITES
cols = list(df_anstat.columns)
cols.remove("Montant brut sans indemnités")
position = cols.index("AUTRES INDEMNITES") + 1
cols.insert(position, "Montant brut sans indemnités")
df_anstat = df_anstat[cols]

print("✅ Colonne 'Montant brut sans indemnités' créée avec succès")
print(f"\nExemples (5 premières lignes) :")
df_anstat[["SECTEUR", "GRADE", "Salaire de Base", "Résidence", "Logement", "Transport",
           "Allocations\n (2 enfants)", "AUTRES INDEMNITES", "Montant brut sans indemnités", "Montant brut"]].head()

✅ Colonne 'Montant brut sans indemnités' créée avec succès

Exemples (5 premières lignes) :


,SECTEUR,GRADE,Salaire de Base,Résidence,Logement,Transport,Allocations\n (2 enfants),AUTRES INDEMNITES,Montant brut sans indemnités,Montant brut
0,Barême Général,A7,548621.60,82293.240,90000,15000,15000,NaN,750914.840,750914.840
1,Barême Général,A7,535781.52,80367.228,90000,15000,15000,NaN,736148.748,736148.748
2,Barême Général,A7,526443.28,78966.492,90000,15000,15000,NaN,725409.772,725409.772
3,Barême Général,A7,506599.52,75989.928,90000,15000,15000,NaN,702589.448,702589.448
4,Barême Général,A7,498428.56,74764.284,90000,15000,15000,NaN,693192.844,693192.844


### Vérification de cohérence

In [4]:
# Vérifier : Montant brut = Montant brut sans indemnités + AUTRES INDEMNITES
df_anstat["AUTRES INDEMNITES_num"] = pd.to_numeric(df_anstat["AUTRES INDEMNITES"], errors='coerce').fillna(0)
df_anstat["check_montant"] = df_anstat["Montant brut sans indemnités"] + df_anstat["AUTRES INDEMNITES_num"]
df_anstat["diff_montant"] = abs(df_anstat["Montant brut"] - df_anstat["check_montant"])

ecarts = df_anstat[df_anstat["diff_montant"] > 0.01]
if len(ecarts) > 0:
    print(f"⚠️  {len(ecarts)} lignes avec un écart entre 'Montant brut' et 'Montant brut sans indemnités + AUTRES INDEMNITES'")
    print(ecarts[["SECTEUR", "GRADE", "Montant brut", "check_montant", "diff_montant"]].head(10))
else:
    print("✅ Cohérence vérifiée : Montant brut = Montant brut sans indemnités + AUTRES INDEMNITES")

# Nettoyage colonnes temporaires
df_anstat.drop(columns=["AUTRES INDEMNITES_num", "check_montant", "diff_montant"], inplace=True)

✅ Cohérence vérifiée : Montant brut = Montant brut sans indemnités + AUTRES INDEMNITES


### 💾 Export du fichier `montant_brut_sans_indemnites`

In [5]:
# Exporter le fichier montant_brut_sans_indemnites
df_export = df_anstat[["SECTEUR", "CATEGORIE", "GRADE", "INDICE", "ECHELON/CLASSE",
                       "Salaire de Base", "Résidence", "Logement", "Transport",
                       "Allocations\n (2 enfants)", "AUTRES INDEMNITES",
                       "Montant brut sans indemnités", "Montant brut"]].copy()

df_export.to_excel("montant_brut_sans_indemnites.xlsx", index=False)
print(f"✅ Fichier 'montant_brut_sans_indemnites.xlsx' exporté ({len(df_export)} lignes)")
df_export.head()

✅ Fichier 'montant_brut_sans_indemnites.xlsx' exporté (754 lignes)


,SECTEUR,CATEGORIE,GRADE,INDICE,ECHELON/CLASSE,Salaire de Base,Résidence,Logement,Transport,Allocations\n (2 enfants),AUTRES INDEMNITES,Montant brut sans indemnités,Montant brut
0,Barême Général,A,A7,2350,21,548621.60,82293.240,90000,15000,15000,NaN,750914.840,750914.840
1,Barême Général,A,A7,2295,33,535781.52,80367.228,90000,15000,15000,NaN,736148.748,736148.748
2,Barême Général,A,A7,2255,32,526443.28,78966.492,90000,15000,15000,NaN,725409.772,725409.772
3,Barême Général,A,A7,2170,31,506599.52,75989.928,90000,15000,15000,NaN,702589.448,702589.448
4,Barême Général,A,A7,2135,44,498428.56,74764.284,90000,15000,15000,NaN,693192.844,693192.844


## 📊 4. Calcul des bornes ANSTAT par Grade

On calcule le **min** et **max** de `Montant brut sans indemnités` pour chaque GRADE dans la grille ANSTAT.

In [6]:
# Min/Max par GRADE (tous secteurs confondus)
df_bornes_grade = (
    df_anstat
    .groupby("GRADE")
    .agg(
        ANSTAT_min=("Montant brut sans indemnités", "min"),
        ANSTAT_max=("Montant brut sans indemnités", "max"),
        ANSTAT_nb_echelons=("Montant brut sans indemnités", "count")
    )
    .reset_index()
    .sort_values("GRADE")
)

print("📋 Bornes ANSTAT par GRADE (Montant brut sans indemnités) :")
print("=" * 65)
df_bornes_grade

📋 Bornes ANSTAT par GRADE (Montant brut sans indemnités) :


,GRADE,ANSTAT_min,ANSTAT_max,ANSTAT_nb_echelons
0,A3,312833.752,761186.000,90
1,A4,340284.588,1051606.168,106
2,A5,379077.796,1109328.164,88
3,A6,473043.836,1357666.984,86
4,A7,646209.824,1461236.680,86
5,B1,241688.036,338338.820,39
6,B3,272562.592,557145.456,90
7,C1,213498.224,275247.336,39
8,C2,217525.340,286589.708,39
9,C3,220210.084,362501.516,52


## 📊 5. Calcul des bornes ANSTAT par Barème × Grade

On calcule le **min** et **max** de `Montant brut sans indemnités` pour chaque combinaison SECTEUR (= Barème) × GRADE.

In [7]:
# Min/Max par SECTEUR + GRADE
df_bornes_bareme_grade = (
    df_anstat
    .groupby(["SECTEUR", "GRADE"])
    .agg(
        ANSTAT_min=("Montant brut sans indemnités", "min"),
        ANSTAT_max=("Montant brut sans indemnités", "max"),
        ANSTAT_nb_echelons=("Montant brut sans indemnités", "count")
    )
    .reset_index()
    .sort_values(["SECTEUR", "GRADE"])
)

print("📋 Bornes ANSTAT par SECTEUR × GRADE (Montant brut sans indemnités) :")
print("=" * 80)
df_bornes_bareme_grade

📋 Bornes ANSTAT par SECTEUR × GRADE (Montant brut sans indemnités) :


,SECTEUR,GRADE,ANSTAT_min,ANSTAT_max,ANSTAT_nb_echelons
0,Barême Général,A3,312833.752,570569.176,13
1,Barême Général,A4,340284.588,662453.868,13
2,Barême Général,A5,379077.796,687823.356,13
3,Barême Général,A6,473043.836,695877.588,13
4,Barême Général,A7,646209.824,750914.840,8
...,...,...,...,...,...
60,para militaire,A7,666209.824,770914.840,8
61,para militaire,B3,292562.592,465728.580,13
62,para militaire,C1,223498.224,271823.616,13
63,para militaire,C2,227525.340,286589.708,13


## 📂 6. Chargement des indicateurs — Feuilles GRADE

On charge toutes les feuilles `GRADE_*` du fichier indicateurs.

In [8]:
xls = pd.ExcelFile(PATH_INDICATEURS)

# Identifier les feuilles GRADE
grade_sheets = [s for s in xls.sheet_names if s.startswith('GRADE_')]
print(f"📋 Feuilles GRADE trouvées : {grade_sheets}")

# Charger et concaténer toutes les feuilles GRADE
dfs_grade = []
for sheet in grade_sheets:
    df = pd.read_excel(xls, sheet_name=sheet, header=None)
    # Les vraies données commencent à la ligne 4 (index 4), header à la ligne 3
    header_row = df.iloc[3].tolist()
    df_data = df.iloc[4:].copy()
    df_data.columns = header_row
    df_data = df_data.reset_index(drop=True)
    # Convertir les colonnes numériques
    for col in ['ANNEE', 'MOIS_NUM', 'Effectif', 'Revenu_moyen', 'Revenu_median',
                'Revenu_min', 'Revenu_max', 'Revenu_p25', 'Revenu_p75']:
        if col in df_data.columns:
            df_data[col] = pd.to_numeric(df_data[col], errors='coerce')
    dfs_grade.append(df_data)

df_indicateurs_grade = pd.concat(dfs_grade, ignore_index=True)
print(f"\n✅ {len(df_indicateurs_grade)} lignes chargées")
print(f"Années : {sorted(df_indicateurs_grade['ANNEE'].unique())}")
print(f"Grades : {sorted(df_indicateurs_grade['GRADE_PRINCIPAL'].unique())}")
df_indicateurs_grade.head()

📋 Feuilles GRADE trouvées : ['GRADE_A3', 'GRADE_A4', 'GRADE_A5', 'GRADE_A6', 'GRADE_A7', 'GRADE_B1', 'GRADE_B3', 'GRADE_C1', 'GRADE_C2', 'GRADE_C3', 'GRADE_D1', 'GRADE_D2']

✅ 288 lignes chargées
Années : [np.int64(2024), np.int64(2025)]
Grades : ['A3', 'A4', 'A5', 'A6', 'A7', 'B1', 'B3', 'C1', 'C2', 'C3', 'D1', 'D2']


,ANNEE,MOIS_NUM,MOIS,GRADE_PRINCIPAL,Effectif,Revenu_moyen,Revenu_median,Revenu_min,Revenu_max,Revenu_p25,Revenu_p75
0,2024,1,Janvier,A3,39166,481451,456838,297834,796187,413883,555015
1,2024,2,Février,A3,40121,489845,463632,297836,798031,416936,564967
2,2024,3,Mars,A3,41029,485310,459338,297836,798106,413883,560570
3,2024,4,Avril,A3,41545,482362,457995,297834,796187,412822,556222
4,2024,5,Mai,A3,41940,480086,456838,297836,796187,408883,553304


## 📂 7. Chargement des indicateurs — Feuilles BAREME × GRADE

On charge les feuilles `BAREME_G_*` qui donnent les indicateurs par Barème × Grade.

In [9]:
# Identifier les feuilles BAREME_G (Barème × Grade)
bareme_g_sheets = [s for s in xls.sheet_names if s.startswith('BAREME_G_')]
print(f"📋 Feuilles BAREME_G trouvées : {bareme_g_sheets}")

# Charger et concaténer
dfs_bareme_g = []
for sheet in bareme_g_sheets:
    df = pd.read_excel(xls, sheet_name=sheet, header=None)
    header_row = df.iloc[3].tolist()
    df_data = df.iloc[4:].copy()
    df_data.columns = header_row
    df_data = df_data.reset_index(drop=True)
    for col in ['ANNEE', 'MOIS_NUM', 'Effectif', 'Revenu_moyen', 'Revenu_median',
                'Revenu_min', 'Revenu_max', 'Revenu_p25', 'Revenu_p75']:
        if col in df_data.columns:
            df_data[col] = pd.to_numeric(df_data[col], errors='coerce')
    dfs_bareme_g.append(df_data)

df_indicateurs_bareme_g = pd.concat(dfs_bareme_g, ignore_index=True)
print(f"\n✅ {len(df_indicateurs_bareme_g)} lignes chargées")
print(f"Barèmes : {sorted(df_indicateurs_bareme_g['bareme'].unique())}")
print(f"Grades  : {sorted(df_indicateurs_bareme_g['GRADE_PRINCIPAL'].unique())}")
df_indicateurs_bareme_g.head()

📋 Feuilles BAREME_G trouvées : ['BAREME_G_Bar_me_G_n_ral', 'BAREME_G_CORPS_DIPL', 'BAREME_G_CORPS_PREF', 'BAREME_G_Cadre_Sup_Sant_', 'BAREME_G_ENS_SEC___PRIM', 'BAREME_G_ENS_SUP', 'BAREME_G_GREFFE', 'BAREME_G_Informaticien', 'BAREME_G_MAGISTRAT_', 'BAREME_G_SECRETAIRE_DIRECTION', 'BAREME_G_Tech_Sant_']

✅ 1311 lignes chargées
Barèmes : ['Barême Général', 'CORPS DIPL', 'CORPS PREF', 'Cadre-Sup-Santé', 'ENS SEC & PRIM', 'ENS SUP', 'GREFFE', 'Informaticien', 'MAGISTRAT ', 'SECRETAIRE DIRECTION', 'Tech-Santé']
Grades  : ['A3', 'A4', 'A5', 'A6', 'A7', 'B1', 'B3', 'C1', 'C2', 'C3', 'D1', 'D2']


,ANNEE,MOIS_NUM,MOIS,bareme,GRADE_PRINCIPAL,Effectif,Revenu_moyen,Revenu_median,Revenu_min,Revenu_max,Revenu_p25,Revenu_p75
0,2024,1,Janvier,Barême Général,A3,8996,383856,382822,297834,598304,325655,424436
1,2024,1,Janvier,Barême Général,A4,4280,461710,450544,325286,692502,373474,528403
2,2024,1,Janvier,Barême Général,A5,1612,615301,621444,382502,720324,575324,670142
3,2024,1,Janvier,Barême Général,A6,433,663894,673379,452677,728379,651581,685879
4,2024,1,Janvier,Barême Général,A7,198,697936,695411,620090,768416,680092,715411


## ✅ 8. Vérification par GRADE

Pour chaque mois, on vérifie si :
- `Revenu_min` des indicateurs ≥ `ANSTAT_min` (le min observé ne peut pas être en dessous de la grille)
- `Revenu_max` des indicateurs ≤ `ANSTAT_max` (le max observé ne peut pas dépasser la grille)

Un salaire **hors** de ces bornes indique un agent dont le salaire ne correspond pas au barème ANSTAT.

In [10]:
# Fusionner indicateurs GRADE avec bornes ANSTAT
df_verif_grade = df_indicateurs_grade.merge(
    df_bornes_grade,
    left_on="GRADE_PRINCIPAL",
    right_on="GRADE",
    how="left"
)

# Vérifications
df_verif_grade["min_OK"] = df_verif_grade["Revenu_min"] >= df_verif_grade["ANSTAT_min"]
df_verif_grade["max_OK"] = df_verif_grade["Revenu_max"] <= df_verif_grade["ANSTAT_max"]
df_verif_grade["ecart_min"] = df_verif_grade["Revenu_min"] - df_verif_grade["ANSTAT_min"]
df_verif_grade["ecart_max"] = df_verif_grade["Revenu_max"] - df_verif_grade["ANSTAT_max"]

# Résumé
nb_total = len(df_verif_grade)
nb_min_ok = df_verif_grade["min_OK"].sum()
nb_max_ok = df_verif_grade["max_OK"].sum()

print("=" * 70)
print("         RÉSULTAT DE VÉRIFICATION PAR GRADE")
print("=" * 70)
print(f"Total lignes vérifiées : {nb_total}")
print(f"Min OK (Revenu_min ≥ ANSTAT_min) : {nb_min_ok}/{nb_total} ({100*nb_min_ok/nb_total:.1f}%)")
print(f"Max OK (Revenu_max ≤ ANSTAT_max) : {nb_max_ok}/{nb_total} ({100*nb_max_ok/nb_total:.1f}%)")
print("=" * 70)

         RÉSULTAT DE VÉRIFICATION PAR GRADE
Total lignes vérifiées : 288
Min OK (Revenu_min ≥ ANSTAT_min) : 50/288 (17.4%)
Max OK (Revenu_max ≤ ANSTAT_max) : 40/288 (13.9%)


### ⚠️ Détail des anomalies MIN par Grade

In [11]:
# Lignes où le min observé est inférieur à la borne ANSTAT
anomalies_min_grade = df_verif_grade[~df_verif_grade["min_OK"]].copy()

if len(anomalies_min_grade) > 0:
    print(f"⚠️  {len(anomalies_min_grade)} mois×grade où Revenu_min < ANSTAT_min :")
    display_cols = ["ANNEE", "MOIS", "GRADE_PRINCIPAL", "Effectif",
                    "Revenu_min", "ANSTAT_min", "ecart_min"]
    print(anomalies_min_grade[display_cols].to_string(index=False))
else:
    print("✅ Aucune anomalie MIN par Grade : tous les Revenu_min ≥ ANSTAT_min")

⚠️  238 mois×grade où Revenu_min < ANSTAT_min :
 ANNEE      MOIS GRADE_PRINCIPAL  Effectif  Revenu_min  ANSTAT_min  ecart_min
  2024   Janvier              A3     39166      297834  312833.752 -14999.752
  2024   Février              A3     40121      297836  312833.752 -14997.752
  2024      Mars              A3     41029      297836  312833.752 -14997.752
  2024     Avril              A3     41545      297834  312833.752 -14999.752
  2024       Mai              A3     41940      297836  312833.752 -14997.752
  2024      Juin              A3     42099      297836  312833.752 -14997.752
  2024   Juillet              A3     43463      297836  312833.752 -14997.752
  2024      Août              A3     44060      297836  312833.752 -14997.752
  2024 Septembre              A3     44346      297834  312833.752 -14999.752
  2024   Octobre              A3     44791      297836  312833.752 -14997.752
  2024  Novembre              A3     45224      297836  312833.752 -14997.752
  2024  Décembre

### ⚠️ Détail des anomalies MAX par Grade

In [12]:
# Lignes où le max observé dépasse la borne ANSTAT
anomalies_max_grade = df_verif_grade[~df_verif_grade["max_OK"]].copy()

if len(anomalies_max_grade) > 0:
    print(f"⚠️  {len(anomalies_max_grade)} mois×grade où Revenu_max > ANSTAT_max :")
    display_cols = ["ANNEE", "MOIS", "GRADE_PRINCIPAL", "Effectif",
                    "Revenu_max", "ANSTAT_max", "ecart_max"]
    print(anomalies_max_grade[display_cols].to_string(index=False))
else:
    print("✅ Aucune anomalie MAX par Grade : tous les Revenu_max ≤ ANSTAT_max")

⚠️  248 mois×grade où Revenu_max > ANSTAT_max :
 ANNEE      MOIS GRADE_PRINCIPAL  Effectif  Revenu_max  ANSTAT_max  ecart_max
  2024   Janvier              A3     39166      796187  761186.000  35001.000
  2024   Février              A3     40121      798031  761186.000  36845.000
  2024      Mars              A3     41029      798106  761186.000  36920.000
  2024     Avril              A3     41545      796187  761186.000  35001.000
  2024       Mai              A3     41940      796187  761186.000  35001.000
  2024      Juin              A3     42099      799008  761186.000  37822.000
  2024   Juillet              A3     43463      797766  761186.000  36580.000
  2024      Août              A3     44060      797766  761186.000  36580.000
  2024 Septembre              A3     44346      797766  761186.000  36580.000
  2024   Octobre              A3     44791      796187  761186.000  35001.000
  2024  Novembre              A3     45224      797766  761186.000  36580.000
  2024  Décembre

### 📊 Synthèse par Grade (vue agrégée)

In [13]:
# Tableau de synthèse : pour chaque grade, nombre de mois en anomalie
synthese_grade = (
    df_verif_grade
    .groupby("GRADE_PRINCIPAL")
    .agg(
        nb_mois=("ANNEE", "count"),
        nb_min_anomalies=("min_OK", lambda x: (~x).sum()),
        nb_max_anomalies=("max_OK", lambda x: (~x).sum()),
        ecart_min_moyen=("ecart_min", lambda x: x[x < 0].mean() if (x < 0).any() else 0),
        ecart_max_moyen=("ecart_max", lambda x: x[x > 0].mean() if (x > 0).any() else 0),
        Revenu_min_global=("Revenu_min", "min"),
        Revenu_max_global=("Revenu_max", "max"),
        ANSTAT_min=("ANSTAT_min", "first"),
        ANSTAT_max=("ANSTAT_max", "first"),
    )
    .reset_index()
)

# Arrondir
for c in ["ecart_min_moyen", "ecart_max_moyen"]:
    synthese_grade[c] = synthese_grade[c].round(0)

print("📊 Synthèse de vérification par GRADE :")
print("=" * 120)
synthese_grade

📊 Synthèse de vérification par GRADE :


,GRADE_PRINCIPAL,nb_mois,nb_min_anomalies,nb_max_anomalies,ecart_min_moyen,ecart_max_moyen,Revenu_min_global,Revenu_max_global,ANSTAT_min,ANSTAT_max
0,A3,24,24,24,-14998.0,36310.0,297834,799008,312833.752,761186.000
1,A4,24,24,24,-14999.0,29211.0,325286,1100803,340284.588,1051606.168
2,A5,24,0,24,0.0,37937.0,382502,1162297,379077.796,1109328.164
3,A6,24,22,24,-18900.0,23337.0,450544,1424543,473043.836,1357666.984
4,A7,24,24,8,-24057.0,53164.0,616213,1529212,646209.824,1461236.680
5,B1,24,24,24,-11945.0,14368.0,229743,354944,241688.036,338338.820
6,B3,24,24,24,-12668.0,27551.0,259793,584961,272562.592,557145.456
7,C1,24,24,24,-10321.0,13024.0,202830,288813,213498.224,275247.336
8,C2,24,24,24,-10065.0,3698.0,206856,290298,217525.340,286589.708
9,C3,24,24,24,-9713.0,17607.0,210210,380605,220210.084,362501.516


## ✅ 9. Vérification par Barème × Grade

Même vérification, mais ventilée par **Barème (SECTEUR)** et **Grade**.
Cela permet de détecter des écarts plus fins propres à un barème spécifique.

In [14]:
# Fusionner indicateurs BAREME_G avec bornes ANSTAT
df_verif_bareme = df_indicateurs_bareme_g.merge(
    df_bornes_bareme_grade,
    left_on=["bareme", "GRADE_PRINCIPAL"],
    right_on=["SECTEUR", "GRADE"],
    how="left"
)

# Vérifications
df_verif_bareme["min_OK"] = df_verif_bareme["Revenu_min"] >= df_verif_bareme["ANSTAT_min"]
df_verif_bareme["max_OK"] = df_verif_bareme["Revenu_max"] <= df_verif_bareme["ANSTAT_max"]
df_verif_bareme["ecart_min"] = df_verif_bareme["Revenu_min"] - df_verif_bareme["ANSTAT_min"]
df_verif_bareme["ecart_max"] = df_verif_bareme["Revenu_max"] - df_verif_bareme["ANSTAT_max"]

# Résumé
nb_total = len(df_verif_bareme.dropna(subset=["ANSTAT_min"]))
nb_sans_ref = df_verif_bareme["ANSTAT_min"].isna().sum()
df_verif_clean = df_verif_bareme.dropna(subset=["ANSTAT_min"])
nb_min_ok = df_verif_clean["min_OK"].sum()
nb_max_ok = df_verif_clean["max_OK"].sum()

print("=" * 70)
print("    RÉSULTAT DE VÉRIFICATION PAR BARÈME × GRADE")
print("=" * 70)
print(f"Total lignes avec référence ANSTAT : {nb_total}")
print(f"Lignes sans référence ANSTAT       : {nb_sans_ref}")
print(f"Min OK (Revenu_min ≥ ANSTAT_min)   : {nb_min_ok}/{nb_total} ({100*nb_min_ok/nb_total:.1f}%)")
print(f"Max OK (Revenu_max ≤ ANSTAT_max)   : {nb_max_ok}/{nb_total} ({100*nb_max_ok/nb_total:.1f}%)")
print("=" * 70)

    RÉSULTAT DE VÉRIFICATION PAR BARÈME × GRADE
Total lignes avec référence ANSTAT : 1311
Lignes sans référence ANSTAT       : 0
Min OK (Revenu_min ≥ ANSTAT_min)   : 295/1311 (22.5%)
Max OK (Revenu_max ≤ ANSTAT_max)   : 277/1311 (21.1%)


### ⚠️ Anomalies MIN par Barème × Grade

In [15]:
anomalies_min_bareme = df_verif_clean[~df_verif_clean["min_OK"]].copy()

if len(anomalies_min_bareme) > 0:
    print(f"⚠️  {len(anomalies_min_bareme)} mois×barème×grade où Revenu_min < ANSTAT_min")
    display_cols = ["ANNEE", "MOIS", "bareme", "GRADE_PRINCIPAL", "Effectif",
                    "Revenu_min", "ANSTAT_min", "ecart_min"]
    # Afficher les 30 premières
    print(anomalies_min_bareme[display_cols].head(30).to_string(index=False))
    if len(anomalies_min_bareme) > 30:
        print(f"\n... et {len(anomalies_min_bareme) - 30} lignes supplémentaires")
else:
    print("✅ Aucune anomalie MIN par Barème × Grade")

⚠️  1016 mois×barème×grade où Revenu_min < ANSTAT_min
 ANNEE    MOIS         bareme GRADE_PRINCIPAL  Effectif  Revenu_min  ANSTAT_min  ecart_min
  2024 Janvier Barême Général              A3      8996      297834  312833.752 -14999.752
  2024 Janvier Barême Général              A4      4280      325286  340284.588 -14998.588
  2024 Janvier Barême Général              A6       433      452677  473043.836 -20366.836
  2024 Janvier Barême Général              A7       198      620090  646209.824 -26119.824
  2024 Janvier Barême Général              B1        44      231688  241688.036 -10000.036
  2024 Janvier Barême Général              B3     13942      260063  272562.592 -12499.592
  2024 Janvier Barême Général              C1      7641      203500  213498.224  -9998.224
  2024 Janvier Barême Général              C2      4638      207526  217525.340  -9999.340
  2024 Janvier Barême Général              C3      6655      210212  220210.084  -9998.084
  2024 Janvier Barême Général       

### ⚠️ Anomalies MAX par Barème × Grade

In [16]:
anomalies_max_bareme = df_verif_clean[~df_verif_clean["max_OK"]].copy()

if len(anomalies_max_bareme) > 0:
    print(f"⚠️  {len(anomalies_max_bareme)} mois×barème×grade où Revenu_max > ANSTAT_max")
    display_cols = ["ANNEE", "MOIS", "bareme", "GRADE_PRINCIPAL", "Effectif",
                    "Revenu_max", "ANSTAT_max", "ecart_max"]
    print(anomalies_max_bareme[display_cols].head(30).to_string(index=False))
    if len(anomalies_max_bareme) > 30:
        print(f"\n... et {len(anomalies_max_bareme) - 30} lignes supplémentaires")
else:
    print("✅ Aucune anomalie MAX par Barème × Grade")

⚠️  1034 mois×barème×grade où Revenu_max > ANSTAT_max
 ANNEE    MOIS         bareme GRADE_PRINCIPAL  Effectif  Revenu_max  ANSTAT_max  ecart_max
  2024 Janvier Barême Général              A3      8996      598304  570569.176  27734.824
  2024 Janvier Barême Général              A4      4280      692502  662453.868  30048.132
  2024 Janvier Barême Général              A5      1612      720324  687823.356  32500.644
  2024 Janvier Barême Général              A6       433      728379  695877.588  32501.412
  2024 Janvier Barême Général              A7       198      768416  750914.840  17501.160
  2024 Janvier Barême Général              B3     13942      466517  445728.580  20788.420
  2024 Janvier Barême Général              C1      7641      274325  261823.616  12501.384
  2024 Janvier Barême Général              C2      4638      290297  276589.708  13707.292
  2024 Janvier Barême Général              C3      6655      320568  306121.892  14446.108
  2024 Janvier Barême Général       

### 📊 Synthèse par Barème × Grade (vue agrégée)

In [17]:
synthese_bareme = (
    df_verif_clean
    .groupby(["bareme", "GRADE_PRINCIPAL"])
    .agg(
        nb_mois=("ANNEE", "count"),
        nb_min_anomalies=("min_OK", lambda x: (~x).sum()),
        nb_max_anomalies=("max_OK", lambda x: (~x).sum()),
        Revenu_min_global=("Revenu_min", "min"),
        Revenu_max_global=("Revenu_max", "max"),
        ANSTAT_min=("ANSTAT_min", "first"),
        ANSTAT_max=("ANSTAT_max", "first"),
    )
    .reset_index()
)

# Filtrer pour ne montrer que ceux avec anomalies
synthese_avec_anomalies = synthese_bareme[
    (synthese_bareme["nb_min_anomalies"] > 0) | (synthese_bareme["nb_max_anomalies"] > 0)
]

if len(synthese_avec_anomalies) > 0:
    print(f"⚠️  {len(synthese_avec_anomalies)} combinaisons Barème×Grade avec au moins une anomalie :")
    print(synthese_avec_anomalies.to_string(index=False))
else:
    print("✅ Aucune anomalie détectée par Barème × Grade")

print(f"\n📊 Tableau complet ({len(synthese_bareme)} combinaisons) :")
synthese_bareme

⚠️  52 combinaisons Barème×Grade avec au moins une anomalie :
              bareme GRADE_PRINCIPAL  nb_mois  nb_min_anomalies  nb_max_anomalies  Revenu_min_global  Revenu_max_global  ANSTAT_min  ANSTAT_max
      Barême Général              A3       24                24                24             297834             599010  312833.752  570569.176
      Barême Général              A4       24                24                24             325286             695250  340284.588  662453.868
      Barême Général              A5       24                 0                24             382502             722210  379077.796  687823.356
      Barême Général              A6       24                 9                24             452677             730180  473043.836  695877.588
      Barême Général              A7       24                23                24             616213             787145  646209.824  750914.840
      Barême Général              B1       24                24           

,bareme,GRADE_PRINCIPAL,nb_mois,nb_min_anomalies,nb_max_anomalies,Revenu_min_global,Revenu_max_global,ANSTAT_min,ANSTAT_max
0,Barême Général,A3,24,24,24,297834,599010,312833.752,570569.176
1,Barême Général,A4,24,24,24,325286,695250,340284.588,662453.868
2,Barême Général,A5,24,0,24,382502,722210,379077.796,687823.356
3,Barême Général,A6,24,9,24,452677,730180,473043.836,695877.588
4,Barême Général,A7,24,23,24,616213,787145,646209.824,750914.840
5,Barême Général,B1,24,24,16,231688,353655,241688.036,338338.820
6,Barême Général,B3,24,24,24,260063,467889,272562.592,445728.580
7,Barême Général,C1,24,24,24,202830,274895,213498.224,261823.616
8,Barême Général,C2,24,24,24,206856,290298,217525.340,276589.708
9,Barême Général,C3,24,24,24,210210,321422,220210.084,306121.892


## 🔗 10. Vérification de correspondance des noms SECTEUR / bareme

On vérifie que les noms de barèmes dans les indicateurs correspondent aux noms de SECTEUR dans ANSTAT.

In [18]:
secteurs_anstat = set(df_anstat["SECTEUR"].unique())
baremes_indicateurs = set(df_indicateurs_bareme_g["bareme"].unique())

print("SECTEURS dans ANSTAT :")
for s in sorted(secteurs_anstat):
    match = "✅" if s in baremes_indicateurs else "❌"
    print(f"  {match} {s}")

print(f"\nBarèmes dans indicateurs :")
for b in sorted(baremes_indicateurs):
    match = "✅" if b in secteurs_anstat else "❌"
    print(f"  {match} {b}")

# Identifier les non-matchés
dans_anstat_pas_indicateurs = secteurs_anstat - baremes_indicateurs
dans_indicateurs_pas_anstat = baremes_indicateurs - secteurs_anstat

if dans_anstat_pas_indicateurs:
    print(f"\n⚠️  SECTEURS dans ANSTAT absents des indicateurs : {dans_anstat_pas_indicateurs}")
if dans_indicateurs_pas_anstat:
    print(f"⚠️  Barèmes dans indicateurs absents d'ANSTAT : {dans_indicateurs_pas_anstat}")
if not dans_anstat_pas_indicateurs and not dans_indicateurs_pas_anstat:
    print("\n✅ Correspondance parfaite entre ANSTAT et indicateurs")

SECTEURS dans ANSTAT :
  ✅ Barême Général
  ✅ CORPS DIPL
  ✅ CORPS PREF
  ✅ Cadre-Sup-Santé
  ✅ ENS SEC & PRIM
  ✅ ENS SUP
  ✅ GREFFE
  ✅ Informaticien
  ✅ MAGISTRAT 
  ✅ SECRETAIRE DIRECTION
  ✅ Tech-Santé
  ❌ para militaire

Barèmes dans indicateurs :
  ✅ Barême Général
  ✅ CORPS DIPL
  ✅ CORPS PREF
  ✅ Cadre-Sup-Santé
  ✅ ENS SEC & PRIM
  ✅ ENS SUP
  ✅ GREFFE
  ✅ Informaticien
  ✅ MAGISTRAT 
  ✅ SECRETAIRE DIRECTION
  ✅ Tech-Santé

⚠️  SECTEURS dans ANSTAT absents des indicateurs : {'para militaire'}


## 💾 11. Export des résultats de vérification

In [ ]:
# Export du résultat complet
with pd.ExcelWriter("resultats_verification_anstat_vs_indicateurs.xlsx", engine="openpyxl") as writer:
    # Synthèse par Grade
    synthese_grade.to_excel(writer, sheet_name="Synthese_par_Grade", index=False)
    
    # Synthèse par Barème × Grade
    synthese_bareme.to_excel(writer, sheet_name="Synthese_Bareme_Grade", index=False)
    
    # Anomalies MIN par Grade
    if len(anomalies_min_grade) > 0:
        cols_export = ["ANNEE", "MOIS", "GRADE_PRINCIPAL", "Effectif",
                       "Revenu_min", "ANSTAT_min", "ecart_min"]
        anomalies_min_grade[cols_export].to_excel(writer, sheet_name="Anomalies_MIN_Grade", index=False)
    
    # Anomalies MAX par Grade
    if len(anomalies_max_grade) > 0:
        cols_export = ["ANNEE", "MOIS", "GRADE_PRINCIPAL", "Effectif",
                       "Revenu_max", "ANSTAT_max", "ecart_max"]
        anomalies_max_grade[cols_export].to_excel(writer, sheet_name="Anomalies_MAX_Grade", index=False)
    
    # Anomalies MIN par Barème × Grade
    if len(anomalies_min_bareme) > 0:
        cols_export = ["ANNEE", "MOIS", "bareme", "GRADE_PRINCIPAL", "Effectif",
                       "Revenu_min", "ANSTAT_min", "ecart_min"]
        anomalies_min_bareme[cols_export].to_excel(writer, sheet_name="Anomalies_MIN_Bareme", index=False)
    
    # Anomalies MAX par Barème × Grade
    if len(anomalies_max_bareme) > 0:
        cols_export = ["ANNEE", "MOIS", "bareme", "GRADE_PRINCIPAL", "Effectif",
                       "Revenu_max", "ANSTAT_max", "ecart_max"]
        anomalies_max_bareme[cols_export].to_excel(writer, sheet_name="Anomalies_MAX_Bareme", index=False)
    
    # Bornes ANSTAT de référence
    df_bornes_grade.to_excel(writer, sheet_name="Ref_ANSTAT_Grade", index=False)
    df_bornes_bareme_grade.to_excel(writer, sheet_name="Ref_ANSTAT_Bareme_Grade", index=False)

print("✅ Résultats exportés dans 'resultats_verification_anstat_vs_indicateurs.xlsx'")
print(f"   Feuilles créées dans le fichier Excel")

## 📝 12. Conclusion

Ce notebook a permis de :

1. ✅ Créer le fichier `montant_brut_sans_indemnites.xlsx` avec la colonne calculée
2. ✅ Vérifier les **min/max par Grade** entre ANSTAT et les indicateurs
3. ✅ Vérifier les **min/max par Barème × Grade** entre ANSTAT et les indicateurs
4. ✅ Exporter les résultats détaillés pour analyse

**Interprétation des résultats** :
- Un `Revenu_min < ANSTAT_min` indique qu'il existe des agents payés **en dessous** de la grille salariale
- Un `Revenu_max > ANSTAT_max` indique qu'il existe des agents payés **au-dessus** de la grille salariale
- Ces écarts peuvent s'expliquer par des primes exceptionnelles, des erreurs de saisie, ou des régimes spéciaux